# Test-like augmentation of the long posts

Reads `long_all.parquet` and writes `long_aug_all.parquet`: the **same rows and folds**, with Telegram-style decoration added:

| augmentation | what | why |
|---|---|---|
| channel footer | appended at the end, with the frequencies seen in the test texts | "Київ — пряма трансляція" etc. are never locations, but a news-trained model tags "Київ" |
| emoji prefix | `❗️ `, `🔥 ` ... at the start | the training text has no emoji at all |
| hashtag line | `#новини` ... at the end | same |

Footers and hashtags are **appended**, so entity offsets do not change; an emoji prefix shifts offsets by its length.
The footer texts and their frequencies were read from the **unlabeled test inputs** (no test labels are used).
Folds are untouched, so CV numbers on `long_aug_all` are comparable between runs, but not with `long_all` (its validation posts now contain footers).

In [1]:
import numpy as np, pandas as pd, random
from pathlib import Path

IN_FILE  = Path("data/splits/long_all.parquet")
OUT_FILE = Path("data/splits/long_aug_all.parquet")
SEED = 42

FOOTER_P   = 0.22     # share of test posts that end with a channel footer (counted on test.csv: about 100 of 477)
EMOJI_P    = 0.15
HASHTAG_P  = 0.05

# (text, weight): weights are the line counts found in test.csv
FOOTERS = [("👉👉Запропонувати новину👈👈", 24),
           ("Хуйовий Київ | Надіслати новину", 20),
           ("Київ — пряма трансляція 👈\nНадіслати новину", 18),
           ("Надіслати новину", 17),
           ("Підписатися    |    Запропонувати новину", 13),
           ("Підписатись | Надіслати новину", 2)]
EMOJI = ["❗️", "🔥", "📍", "🚨", "‼️", "⚡️", "🔴", "💥", "🚧", "🆘", "❓"]
HASHTAGS = ["#новини", "#афіша", "#важливо", "#війна", "#допомога", "#ЗСУ", "#оголошення"]

In [2]:
df = pd.read_parquet(IN_FILE)
print(len(df), "posts;", df["split"].value_counts().to_dict())

rnd = random.Random(SEED)
foot_texts, foot_w = zip(*FOOTERS)

def shift(markers, d):
    return [[int(x[0]) + d, int(x[1]) + d] for x in markers]      # parquet gives arrays of arrays

texts, locs, orgs, pers, kinds = [], [], [], [], []
for t, lm, om, pm in zip(df["text"], df["loc_markers"], df["org_markers"], df["per_markers"]):
    d, kind = 0, []
    if rnd.random() < EMOJI_P and not (t and ord(t[0]) > 0x2000):          # do not double-decorate
        pre = rnd.choice(EMOJI) + " "
        t, d = pre + t, len(pre); kind.append("emoji")
    if rnd.random() < HASHTAG_P:
        t += "\n\n" + " ".join(rnd.sample(HASHTAGS, rnd.choice([1, 2]))); kind.append("hashtag")
    if rnd.random() < FOOTER_P:
        t += rnd.choice(["\n\n", "\n"]) + rnd.choices(foot_texts, foot_w)[0]; kind.append("footer")
    texts.append(t); kinds.append(",".join(kind))
    locs.append(shift(lm, d)); orgs.append(shift(om, d)); pers.append(shift(pm, d))

out = df.copy()
out["text"], out["loc_markers"], out["org_markers"], out["per_markers"], out["aug"] = texts, locs, orgs, pers, kinds
out["locations"] = [[t[a:b] for a, b in m] for t, m in zip(out["text"], out["loc_markers"])]
assert [list(x) for x in df["locations"]] == [list(x) for x in out["locations"]], "offsets were shifted incorrectly"
out.to_parquet(OUT_FILE, index=False)
print("saved", OUT_FILE)

886195 posts; {'dev': 877410, 'holdout': 8785}
saved data\splits\long_aug_all.parquet


In [3]:
# checks
for k in ("emoji", "hashtag", "footer"):
    print(f"{k:8s} {out['aug'].str.contains(k).mean():.1%}")
print("text length quantiles:", np.percentile(out["text"].str.len(), [10, 25, 50, 75, 90, 99]).round())
ex = out[out["aug"].str.contains("footer") & (out["n_loc"] > 0)].iloc[0]
print(repr(ex["text"][:200]), "...", repr(ex["text"][-80:]))
print("locations:", list(ex["locations"]))

emoji    12.8%
hashtag  5.0%
footer   22.0%
text length quantiles: [ 118.  183.  302.  515.  786. 1587.]
'🚨 Однак до його настанов волейболістки так і не дослухалися.\n\nЗазначимо, у верхній палаті американського парламенту розгляд резолюції відбувається подібно до судового засідання – сторони можуть заслух' ... 'остійній основі працюють пункти збору батарейок.\nХуйовий Київ | Надіслати новину'
locations: ['Ташкенті', 'Нідерландів', 'Старосіллі', 'Львова']


Next: in `validation_loop.ipynb` set `DATA_FILE = "data/splits/long_aug_all.parquet"` and `SUFFIX = "_aug"`, then run the whole notebook.